## Setup paths

In [ ]:
from pathlib import Path
import os
import sys

# Project root: assumes notebook is run from notebooks/
PROJECT_ROOT = Path.cwd().resolve().parent

# Main folders
SRC_DIR = PROJECT_ROOT / "src"
PACKAGE_DIR = SRC_DIR / "brain_mesh_creation"
wrk_dir = str(SRC_DIR / "dependencies")
sub_dir = str(PROJECT_ROOT / "data" / "subjects")

# Subject and output names
subject_folder = "TBI-generic-50perc-avg-male"
output_name = "output"

# Make sure src/ is importable
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from brain_mesh_creation import bmctk
from brain_mesh_creation import mesh_utils

print("PROJECT_ROOT:", PROJECT_ROOT)
print("wrk_dir:", wrk_dir)
print("sub_dir:", sub_dir)
print("subject_folder:", subject_folder)

## Meshing parameters setup

In [ ]:
# Set mesh size (in mm)
mesh_size = 1.5

# Voxel correction variables
# It is generally recommended not to change these unless needed
iterations = 3
blocks = 1
threshold = 0.9  # Threshold for neighbour disagreement during voxel correction

# Manual centre-of-gravity edits
# Used for generating the centre of gravity for TBI simulations
cog_id = 80000000
change_ap = 0
change_lr = 0
change_is = 0
change_cog = [change_ap, change_lr, change_is]

## Mesh creation

In [ ]:
import os

subject_path = os.path.join(sub_dir, subject_folder)
pre_model_path = os.path.join(subject_path, "pre_model.nii.gz")

print("Looking at:", subject_path)

if os.path.isdir(subject_path):

    if not os.path.isfile(pre_model_path):
        print(f"Cannot find pre_model.nii.gz at: {pre_model_path}")
        print("Please run notebook 01 first to generate pre_model.nii.gz")
    else:
        print("Starting mesh generation for", subject_folder)

        # Move into subject folder so output files are written there
        os.chdir(subject_path)
        print("Working directory:", os.getcwd())

        # Initialise model creation
        my_model = bmctk.create_model("pre_model.nii.gz", output_name, wrk_dir)

        # Create relative coordinates
        my_model.create_coordinates()

        # Resample model if necessary
        my_model.resample(mesh_size)

        # Assign missing grey matter
        my_model.gm_check()

        # Check contact of grey matter to skull
        my_model.check_contact(257, 256, 42)
        my_model.check_contact(257, 256, 3)
        my_model.check_contact(257, 256, 47)
        my_model.check_contact(257, 256, 8)

        # Check contact of CSF to void
        my_model.check_contact(0, 257, 256)

        # Check contact of CSF to white matter
        my_model.check_contact(41, 42, 256)
        my_model.check_contact(2, 3, 256)
        my_model.check_contact(46, 47, 256)
        my_model.check_contact(7, 8, 256)

        # Make corrections for voxels inside other parts
        my_model.voxel_corrections(iterations, threshold, blocks)

        # Create falx
        LR_center_coords = my_model.create_falx()
        LR_center = LR_center_coords[0]

        # Create tentorium
        my_model.create_tentorium()

        # Create mesh
        max_node = my_model.create_k_mesh()
        print("Largest node number assigned:", max_node)

        # Create set list
        my_model.set_list()

        print("Finished making model for", subject_folder)
        print()
else:
    print("Cannot see folder", subject_folder, "in", sub_dir)

## Mesh smoothing

In [ ]:
%%bash -s "$mesh_size" "$wrk_dir" "$sub_dir" "$subject_folder" "$output_name"

set -e

MESH_SIZE="$1"
WRK_DIR="$2"
SUB_DIR="$3"
SUBJECT="$4"
OUTPUT_NAME="$5"

SUBJECT_PATH="${SUB_DIR}/${SUBJECT}"
OUTPUT_PATH="${SUBJECT_PATH}/${OUTPUT_NAME}"
AOUT_PATH="${WRK_DIR}/rs/a.out"

echo "Looking at ${SUBJECT}"
cd "${SUBJECT_PATH}" || exit 1
pwd
echo "Moved to ${SUBJECT}"

if [ ! -f "${AOUT_PATH}" ]; then
    echo "Cannot find smoothing executable: ${AOUT_PATH}"
    exit 1
fi

if [ ! -f "${OUTPUT_PATH}/mesh.k" ]; then
    echo "Cannot find input mesh: ${OUTPUT_PATH}/mesh.k"
    exit 1
fi

cd "${OUTPUT_PATH}" || exit 1
echo "Running mesh smoothing in $(pwd)"

"${AOUT_PATH}" mesh.k mesh_smoothed.k 8 0.2 200 2000 "${MESH_SIZE}"

echo "Created: ${OUTPUT_PATH}/mesh_smoothed.k"
cd ..

## Brain-Skull Contact Prevention

In [ ]:
import os
import sys
from pathlib import Path

# Make sure src/ is importable
PROJECT_ROOT = Path.cwd().resolve().parent
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from brain_mesh_creation import mesh_utils

current_subject_dir = os.path.join(sub_dir, subject_folder)
mesh_output_dir = os.path.join(current_subject_dir, output_name)

input_k_file = os.path.join(mesh_output_dir, "mesh_smoothed.k")
output_k_file = os.path.join(mesh_output_dir, "mesh_smoothed_revised.k")

print(f"--- Processing Subject: {subject_folder} ---")
print("Applying brain-skull interface regularization...")
print("Input:", input_k_file)
print("Output:", output_k_file)

if not os.path.isfile(input_k_file):
    print(f"Cannot find input mesh file: {input_k_file}")
else:
    mesh_utils.create_csf_buffer(input_k_file, output_k_file)
    print("Finished creating revised mesh:")
    print(output_k_file)